# Ejercicios ensembling
En este ejercicio vas a realizar prediciones sobre un dataset de ciudadanos indios diabéticos. Se trata de un problema de clasificación en el que intentaremos predecir 1 (diabético) 0 (no diabético).

### 1. Carga las librerias que consideres comunes al notebook

In [3]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import seaborn as sns
from sklearn.model_selection import KFold, cross_val_score, train_test_split, GridSearchCV, cross_validate


### 2. Lee los datos de [esta direccion](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv)
Los nombres de columnas son:
```Python
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
```

In [4]:
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

df = pd.read_csv(url, names=names)

In [5]:
df.head()

,preg,plas,pres,skin,test,mass,pedi,age,class
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


### 3. Bagging
Para este apartado tendrás que crear un ensemble utilizando la técnica de bagging ([BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)), mediante la cual combinarás 100 [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html). Recuerda utilizar también [cross validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) con 10 kfolds.

**Para este apartado y siguientes, no hace falta que dividas en train/test**, por hacerlo más sencillo. Simplemente divide tus datos en features y target.

Establece una semilla

In [6]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# Separación en features y target
X = df.drop('class', axis=1)
y = df['class']

# Configuración de semilla y validación cruzada
seed = 42
kfold = KFold(n_splits=10, shuffle=True, random_state=seed)
scoring = ['accuracy', 'recall']

# Modelo Bagging con 100 árboles de decisión
estimator = DecisionTreeClassifier(random_state=seed)
model_bagging = BaggingClassifier(estimator=estimator, n_estimators=100, random_state=seed)

res_bagging = cross_validate(model_bagging, X, y, cv=kfold, scoring=scoring)
print(f"Bagging -> Accuracy: {res_bagging['test_accuracy'].mean():.4f} | Recall: {res_bagging['test_recall'].mean():.4f}")

Bagging -> Accuracy: 0.7565 | Recall: 0.5963


### 4. Random Forest
En este caso entrena un [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) con 100 árboles y un `max_features` de 3. También con validación cruzada

In [7]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators=100, max_features=3, random_state=seed)
res_rf = cross_validate(model_rf, X, y, cv=kfold, scoring=scoring)
print(f"Random Forest -> Accuracy: {res_rf['test_accuracy'].mean():.4f} | Recall: {res_rf['test_recall'].mean():.4f}")

Random Forest -> Accuracy: 0.7695 | Recall: 0.5996


### 5. AdaBoost
Implementa un [AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) con 30 árboles.

In [8]:
from sklearn.ensemble import AdaBoostClassifier

model_ada = AdaBoostClassifier(n_estimators=30, random_state=seed)
res_ada = cross_validate(model_ada, X, y, cv=kfold, scoring=scoring)
print(f"AdaBoost -> Accuracy: {res_ada['test_accuracy'].mean():.4f} | Recall: {res_ada['test_recall'].mean():.4f}")

AdaBoost -> Accuracy: 0.7617 | Recall: 0.5833


### 6. GradientBoosting
Implementa un [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) con 100 estimadores

In [9]:
from sklearn.ensemble import GradientBoostingClassifier

model_gb = GradientBoostingClassifier(n_estimators=100, random_state=seed)
res_gb = cross_validate(model_gb, X, y, cv=kfold, scoring=scoring)
print(f"Gradient Boosting -> Accuracy: {res_gb['test_accuracy'].mean():.4f} | Recall: {res_gb['test_recall'].mean():.4f}")

Gradient Boosting -> Accuracy: 0.7668 | Recall: 0.6138


### 7. XGBoost
Para este apartado utiliza un [XGBoostClassifier](https://docs.getml.com/latest/api/getml.predictors.XGBoostClassifier.html) con 100 estimadores. XGBoost no forma parte de la suite de modelos de sklearn, por lo que tendrás que instalarlo con pip install

In [10]:
# !pip install xgboost
from xgboost import XGBClassifier

model_xgb = XGBClassifier(n_estimators=100, random_state=seed)
res_xgb = cross_validate(model_xgb, X, y, cv=kfold, scoring=scoring)
print(f"XGBoost -> Accuracy: {res_xgb['test_accuracy'].mean():.4f} | Recall: {res_xgb['test_recall'].mean():.4f}")

XGBoost -> Accuracy: 0.7409 | Recall: 0.6039


### 8. Primeros resultados
Crea un dataframe con los resultados y sus algoritmos, ordenándolos de mayor a menor

In [11]:
resultados = pd.DataFrame({
    'Algoritmo': ['Bagging', 'Random Forest', 'AdaBoost', 'Gradient Boosting', 'XGBoost'],
    'Accuracy': [
        res_bagging['test_accuracy'].mean(),
        res_rf['test_accuracy'].mean(),
        res_ada['test_accuracy'].mean(),
        res_gb['test_accuracy'].mean(),
        res_xgb['test_accuracy'].mean()
    ],
    'Recall': [
        res_bagging['test_recall'].mean(),
        res_rf['test_recall'].mean(),
        res_ada['test_recall'].mean(),
        res_gb['test_recall'].mean(),
        res_xgb['test_recall'].mean()
    ]
})

resultados = resultados.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
resultados

,Algoritmo,Accuracy,Recall
0,Random Forest,0.769463,0.599567
1,Gradient Boosting,0.766815,0.613840
2,AdaBoost,0.761688,0.583322
3,Bagging,0.756459,0.596309
4,XGBoost,0.740858,0.603879


### 9. Hiperparametrización
Vuelve a entrenar los modelos de nuevo, pero esta vez dividiendo el conjunto de datos en train/test y utilizando un gridsearch para encontrar los mejores hiperparámetros.

In [12]:
from sklearn.metrics import accuracy_score, recall_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

modelos_params = {
    'Bagging': (
        BaggingClassifier(estimator=DecisionTreeClassifier(), random_state=seed),
        {'n_estimators': [50, 100, 150], 'max_samples': [0.7, 1.0]}
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=seed),
        {'n_estimators': [50, 100, 200], 'max_features': [2, 3, 4], 'max_depth': [None, 5, 10]}
    ),
    'AdaBoost': (
        AdaBoostClassifier(random_state=seed),
        {'n_estimators': [30, 50, 100], 'learning_rate': [0.01, 0.1, 1.0]}
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=seed),
        {'n_estimators': [50, 100, 150], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]}
    ),
    'XGBoost': (
        XGBClassifier(random_state=seed),
        {'n_estimators': [50, 100, 150], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]}
    )
}

resultados_grid = []

for nombre, (modelo, params) in modelos_params.items():
    # refit='accuracy' (cámbialo a 'recall' si quieres optimizar directamente la sensibilidad)
    grid = GridSearchCV(modelo, params, cv=5, scoring=['accuracy', 'recall'], refit='recall', n_jobs=-1)
    grid.fit(X_train, y_train)
    
    # Predicción con el mejor estimador encontrado
    y_pred = grid.best_estimator_.predict(X_test)
    
    resultados_grid.append({
        'Algoritmo': nombre,
        'CV Accuracy': grid.cv_results_['mean_test_accuracy'][grid.best_index_],
        'CV Recall': grid.cv_results_['mean_test_recall'][grid.best_index_],
        'Test Accuracy': accuracy_score(y_test, y_pred),
        'Test Recall': recall_score(y_test, y_pred),
        'Mejores Parámetros': grid.best_params_
    })

df_grid = pd.DataFrame(resultados_grid).sort_values(by='Test Accuracy', ascending=False)
df_grid

,Algoritmo,CV Accuracy,CV Recall,Test Accuracy,Test Recall,Mejores Parámetros
2,AdaBoost,0.775317,0.615393,0.779221,0.727273,"{'learning_rate': 1.0, 'n_estimators': 50}"
4,XGBoost,0.781781,0.643522,0.753247,0.654545,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti..."
3,Gradient Boosting,0.770399,0.624917,0.753247,0.672727,"{'learning_rate': 0.05, 'max_depth': 3, 'n_est..."
1,Random Forest,0.771971,0.615393,0.746753,0.672727,"{'max_depth': 10, 'max_features': 4, 'n_estima..."
0,Bagging,0.773624,0.619934,0.740260,0.672727,"{'max_samples': 1.0, 'n_estimators': 100}"


### 10. Conclusiones finales

He usado recall como prioridad porque me interesa que aunque haya falsos positivos se hagan pruebas aún así (para no dejar a ningún potencial diabético sin diagnosticar).

Sin hiperparámetros se obtiene el mejor recall para gradient boosting y con hiperparámetros se obtiene el mejor test recall para ada boost con los hiperparámetros {'learning_rate': 1.0, 'n_estimators': 50}.